# 관광데이터랩에서 분류 구분해서 저장하기 (추후 페르소나별 키워드를 위한 근거)

#### 노가다... 크롤링이 안됨 (https://datalab.visitkorea.or.kr/datalab/portal/loc/getTourDataFormBeta.do)

#### 인문 (문화/예술/역사)는 각각을 대분류로 쪼갤 것

#### 대표 키워드는 해당 링크에서 한국관광100선으로 필터링한것

In [ ]:
import pandas as pd

data = [
    # 대분류: 역사
    {"대분류": "역사", 
     "중분류": "역사관광지", 
     "소분류": "사찰, 유적지/사적지, 고궁, 문, 민속마을, 고택, 생가, 성", 
     "대표키워드": "경복궁, 경희궁, 덕수궁, 창경궁, 창덕궁, 후원", 
     "연관페르소나": "문화유산 탐험가"},

    # 대분류: 휴양
    {"대분류": "휴양", 
     "중분류": "휴양관광지", 
     "소분류": "공원, 테마공원, 유람선/잠수함관광, 관광단지", 
     "대표키워드": "서울숲", 
     "연관페르소나": ""},

    # 대분류: 문화/예술
    # 공연장은 추후 kpop 테마를 위해..제거
    {"대분류": "문화/예술", 
     "중분류": "문화시설", 
     "소분류": "기념관, 박물관, 컨벤션센터, 전시관, 미술관/화랑, 문화전수시설, 문화원", 
     "대표키워드": "서대문형무소역사관, 서울시립미술관, 한국종합무역센터", 
     "연관페르소나": "플렉스 장기 체류객"},

    # 대분류: 체험
    {"대분류": "체험", 
     "중분류": "체험관광지", 
     "소분류": "이색거리, 이색체험, 전통체험", 
     "대표키워드": "서촌마을, 청와대", 
     "연관페르소나": ""},

    # 대분류: 건축/조형물
    {"대분류": "건축/조형물", 
     "중분류": "건축/조형물", 
     "소분류": "유명건물, 기념탑/기념비/전망대", 
     "대표키워드": "남산서울타워", 
     "연관페르소나": "알뜰 필수코스 정복자"},


    # 대분류: 자연
    {"대분류": "자연", 
     "중분류": "자연관광지", 
     "소분류": "자연생태관광지, 산, 수목원, 계곡, 항구/포구", 
     "대표키워드": "한강, 탄천", 
     "연관페르소나": ""},

    # 대분류: 쇼핑
    {"대분류": "쇼핑", 
     "중분류": "쇼핑", 
     "소분류": "상설시장, 전문상가", 
     "대표키워드": "", 
     "연관페르소나": "실속파 쇼핑 원정대"},
]

df = pd.DataFrame(data)
df.to_csv("분류_페르소나_매핑표.csv", index=False, encoding='utf-8-sig')
print("✅ 저장 완료: 분류_페르소나_매핑표.csv")

## 매핑매핑

In [5]:
import pandas as pd
import ast

# =========================
# 1. 매핑표 불러오기
# =========================
mapping_df = pd.read_csv("분류_페르소나_매핑표.csv", encoding="utf-8-sig")

# 소분류/대표키워드/중분류/대분류를 기준으로 persona 매핑 리스트 생성
mapping_rules = []
for _, row in mapping_df.iterrows():
    persona = row['연관페르소나']

    # 대분류
    if pd.notna(row['대분류']):
        mapping_rules.append((str(row['대분류']).strip(), persona, "대"))

    # 중분류
    if pd.notna(row['중분류']):
        mapping_rules.append((str(row['중분류']).strip(), persona, "중"))

    # 소분류
    if pd.notna(row['소분류']):
        mapping_rules.append((str(row['소분류']).strip(), persona, "소"))

    # 대표키워드 (쉼표로 여러 개일 수 있음)
    if pd.notna(row['대표키워드']):
        for kw in str(row['대표키워드']).split(","):
            mapping_rules.append((kw.strip(), persona, "대표"))

# =========================
# 2. POI 데이터 불러오기
# =========================
poi_df = pd.read_csv("서울POI풀_키워드추출.csv", encoding="utf-8-sig")

# top_keywords 문자열 → 리스트 변환
def parse_keywords(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [w.strip() for w in x.split(",")]
    elif isinstance(x, list):
        return x
    return []

poi_df['parsed_keywords'] = poi_df['top_keywords'].apply(parse_keywords)

# =========================
# 3. 매핑 함수 정의 (존재 기반, 부분매칭)
# =========================
def map_personas(keywords, desc):
    personas_found = []
    desc = str(desc)

    for term, persona, lvl in mapping_rules:
        # top_keywords 또는 설명에 부분일치하면 매핑
        kw_match = any(term in str(kw) for kw in keywords)
        desc_match = term in desc

        if kw_match or desc_match:
            if persona not in personas_found:
                personas_found.append(persona)

    return personas_found

# =========================
# 4. 매핑 적용
# =========================
poi_df['persona_list'] = poi_df.apply(
    lambda row: map_personas(row['parsed_keywords'], row['설명']), axis=1
)

poi_df['top1_persona'] = poi_df['persona_list'].apply(lambda x: x[0] if len(x) > 0 else None)
poi_df['top2_persona'] = poi_df['persona_list'].apply(lambda x: x[1] if len(x) > 1 else None)
poi_df['top3_persona'] = poi_df['persona_list'].apply(lambda x: x[2] if len(x) > 2 else None)

# 대표 persona = top1
poi_df['persona'] = poi_df['top1_persona']

# =========================
# 5. 컬럼 순서 정렬
# =========================
priority = ['명소명', '설명', 'top_keywords', 'persona', '평점', '리뷰수', '출처']
for col in priority:
    if col not in poi_df.columns:
        poi_df[col] = pd.NA  # 없으면 빈 값 채우기

others = [c for c in poi_df.columns if c not in priority]
ordered_cols = [c for c in priority if c in poi_df.columns] + others
poi_df = poi_df[ordered_cols]

# =========================
# 6. 결과 저장
# =========================
poi_df.to_csv("서울POI_페르소나매핑.csv", index=False, encoding="utf-8-sig")
print("✅ 저장 완료: 서울POI_페르소나매핑.csv")

✅ 저장 완료: 서울POI_페르소나매핑.csv


##### 1. 리뷰수 30개 미만인거 버리기
##### 2. 위치 겹치는거 버리기 (ex. 롯데백화점 본점 & 롯데백화점 영플라자 명동)
##### 3. 호텔, 음식점 버리기
##### 4. 그 외.. 인간의 눈으로 수동 매핑
